# Qversity Fintech — Análisis de Preguntas de Negocio Clave

**Pipeline:** ELT end-to-end migrado a Databricks Free Edition  
**Stack:** PySpark (bronze → silver_raw) · dbt-databricks (silver → gold) · Unity Catalog  
**Dataset:** ~5.100 clientes sintéticos, 7 países LATAM, 87.686 transacciones  

Este notebook analiza **una pregunta de negocio por cada página del dashboard de Power BI**:

| # | Dashboard | Pregunta |
|---|---|---|
| 1 | Revenue & Transactions | ¿Qué canales concentran más volumen y cuál es su confiabilidad operacional? |
| 2 | Risk & Credit | ¿En qué segmento de clientes se concentra el riesgo de delinquencia y default? |
| 3 | Customer & Engagement | ¿Existe estacionalidad o tendencia en la adquisición mensual de clientes? |

> **Fuente:** `workspace.gold.*` (Unity Catalog, Databricks Free Edition)

In [ ]:
CATALOG = "workspace"
spark.sql(f"USE CATALOG {CATALOG}")
print(f"Catálogo activo: {CATALOG}")

---
# Pregunta 1 — Revenue & Transactions
## ¿Qué canales concentran más volumen transaccional y cuál es su confiabilidad operacional?

### Contexto de negocio

En una institución financiera LATAM, los canales de transacción (mobile, web, ATM, branch, POS)
no solo difieren en volumen sino en **perfil de riesgo operacional**. Un canal de alto volumen
con alta tasa de fallo implica pérdida de ingresos, fricción en la experiencia del cliente y
costos de resolución de disputas.

La pregunta estratégica no es "¿qué canal mueve más plata?" sino
**"¿qué canal mueve más plata con menor riesgo de fallo?"**

Para responderla construimos un **índice de eficiencia operacional** que combina volumen
realizado (transacciones completadas) con penalización por fallo, permitiendo rankear
los canales en una sola métrica comparable.

**Mart:** `gold.mart_tx_by_channel` · **Grain:** (channel, currency)

In [ ]:
df_channels = spark.sql("""
    SELECT
        channel,
        SUM(tx_count)                                               AS total_intentos,
        SUM(completed_tx_count)                                     AS total_completadas,
        SUM(failed_tx_count)                                        AS total_fallidas,
        ROUND(SUM(failed_tx_count) * 100.0
              / NULLIF(SUM(tx_count), 0), 2)                        AS tasa_fallo_pct,
        ROUND(SUM(completed_tx_count) * 100.0
              / NULLIF(SUM(tx_count), 0), 2)                        AS tasa_completacion_pct,
        ROUND(AVG(avg_ticket), 2)                                   AS ticket_promedio_usd,
        -- Índice de eficiencia: volumen completado ponderado por tasa de éxito
        -- Un canal con alto volumen pero alta tasa de fallo baja en el ranking
        ROUND(
            SUM(completed_tx_count) * 1.0
            / NULLIF(SUM(tx_count), 0)
            * SUM(tx_count) / 1000, 1
        )                                                           AS indice_eficiencia,
        RANK() OVER (ORDER BY
            SUM(completed_tx_count) * 1.0
            / NULLIF(SUM(tx_count), 0)
            * SUM(tx_count) DESC
        )                                                           AS ranking
    FROM gold.mart_tx_by_channel
    GROUP BY channel
    ORDER BY ranking
""")

display(df_channels)

### Hallazgos

**Tasa de fallo ~25% en todos los canales** — muy por encima del benchmark real de la industria
(< 3%). El generador sintético distribuyó los estados de transacción uniformemente entre
`completed / pending / failed / reversed` (~25% cada uno). En una operación real, esperaríamos
> 95% de completación. Este hallazgo está documentado en `decisions.md`.

**Volúmenes homogéneos entre canales** (~17.5K transacciones cada uno). En una banca real,
`mobile` y `web` concentrarían más volumen que `branch` o `atm` por el crecimiento del canal
digital en LATAM. La homogeneidad confirma que el generador no correlacionó el canal de
preferencia del cliente con el tipo de transacción.

**Recomendación operativa:** Aunque el índice de eficiencia no diferencia significativamente
los canales en este dataset, en producción este mismo indicador permitiría priorizar
inversión en confiabilidad donde el ticket promedio es más alto — un fallo en `branch`
(operaciones de mayor valor) impacta más revenue que un fallo en `atm`.

---
# Pregunta 2 — Risk & Credit
## ¿En qué segmento de clientes se concentra el riesgo de delinquencia y default?

### Contexto de negocio

La gestión del riesgo crediticio requiere segmentar la cartera para identificar
dónde concentrar recursos de cobranza, dónde endurecer criterios de originación
y dónde existe potencial de cross-sell sin aumentar el riesgo sistémico.

La pregunta combina tres dimensiones:
- **Delinquency rate** (DPD ≥ 30): señal temprana de deterioro
- **Default rate** (DPD ≥ 90 o status = 'default'): exposición a pérdida real (Basel/IFRS9)
- **Credit score bucket**: calidad crediticia inicial que debería predecir el fallo

Si el modelo de originación funciona bien, `private_banking` debería tener menor
delinquencia que `retail`. Si no hay diferencia, o el modelo falla o el dato es sintético.

**Marts:** `gold.mart_delinquency_by_segment` + `gold.mart_customer_360`

In [ ]:
df_risk = spark.sql("""
    SELECT
        d.customer_segment,
        d.customer_count,
        d.delinquent_customers,
        ROUND(d.delinquency_rate * 100, 2)          AS delinquency_rate_pct,
        ROUND(d.default_rate * 100, 2)              AS default_rate_pct,
        -- Clientes en banda crediticia Poor (FICO < 580): proxy de fragilidad
        COUNT(CASE WHEN c.credit_score < 580
                    AND c.credit_score IS NOT NULL
                   THEN 1 END)                      AS clientes_poor_credit,
        ROUND(
            COUNT(CASE WHEN c.credit_score < 580
                        AND c.credit_score IS NOT NULL
                       THEN 1 END) * 100.0
            / NULLIF(d.customer_count, 0), 1
        )                                           AS pct_poor_credit,
        ROUND(AVG(c.credit_score), 0)               AS score_promedio,
        -- Brecha entre delinquency real y la esperada por score
        -- En un modelo bien calibrado, mayor pct_poor debería implicar mayor delinquency
        ROUND(
            d.delinquency_rate * 100 - (
                COUNT(CASE WHEN c.credit_score < 580
                            AND c.credit_score IS NOT NULL
                           THEN 1 END) * 100.0
                / NULLIF(d.customer_count, 0)
            ), 2
        )                                           AS brecha_delinquency_vs_poor_credit
    FROM gold.mart_delinquency_by_segment d
    LEFT JOIN gold.mart_customer_360 c
        ON c.customer_segment = d.customer_segment
    GROUP BY
        d.customer_segment,
        d.customer_count,
        d.delinquent_customers,
        d.delinquency_rate,
        d.default_rate
    ORDER BY d.delinquency_rate DESC
""")

display(df_risk)

### Hallazgos

**Delinquencia ~63% y default ~25% uniformes entre segmentos** — en una cartera real,
`retail` mostraría delinquencia 2-3x mayor que `private_banking`. La uniformidad es un
artefacto del generador que no correlacionó el estado del préstamo con el segmento.

**La columna `brecha_delinquency_vs_poor_credit`** mide si la tasa de clientes Poor
explica la delinquencia observada. En un modelo bien calibrado, la brecha debería ser
cercana a cero. En este dataset, la brecha es alta porque ambas variables (delinquencia
y credit score) fueron generadas independientemente sin correlación.

**46% de clientes en banda Poor (score < 580)** con score promedio de 573 — documentado
desde el EDA del día 1. En una cartera real, esto indicaría una política de originación
muy permisiva o un sesgo de selección adversa severo.

**Hipótesis FICO no se sostiene:** La utilización no predice el fallo en este dataset
(todos los buckets muestran ~63% de delinquencia). Documentado en `decisions.md`
bajo "Key findings".

---
# Pregunta 3 — Customer & Engagement
## ¿Existe estacionalidad o tendencia en la adquisición mensual de clientes?

### Contexto de negocio

Entender si la adquisición tiene patrones estacionales permite optimizar el presupuesto
de marketing (concentrar inversión en meses de alta demanda natural), dimensionar recursos
de onboarding y evaluar efectividad de campañas aislando el efecto estacional base.

El análisis responde tres preguntas en una:
1. ¿Hay meses consistentemente mejores o peores? (estacionalidad)
2. ¿La base crece año a año? (tendencia)
3. ¿El MoM growth es predecible o aleatorio? (volatilidad)

Para detectar estacionalidad calculamos el **coeficiente de variación** por mes del año:
si es bajo y consistente entre años, hay patrón; si es alto y errático, la distribución
es aleatoria.

**Mart:** `gold.mart_acquisition_trend` · **Grain:** mes calendario

In [ ]:
df_estacionalidad = spark.sql("""
    WITH por_mes_anio AS (
        SELECT
            year,
            month_number,
            new_customers,
            mom_growth_pct
        FROM gold.mart_acquisition_trend
        -- Excluir mayo 2026: mes incompleto (17 clientes = artefacto de corte del dataset)
        WHERE NOT (year = 2026 AND month_number = 5)
    ),
    estacionalidad AS (
        SELECT
            month_number,
            CASE month_number
                WHEN 1  THEN '01-Enero'      WHEN 2  THEN '02-Febrero'
                WHEN 3  THEN '03-Marzo'      WHEN 4  THEN '04-Abril'
                WHEN 5  THEN '05-Mayo'       WHEN 6  THEN '06-Junio'
                WHEN 7  THEN '07-Julio'      WHEN 8  THEN '08-Agosto'
                WHEN 9  THEN '09-Septiembre' WHEN 10 THEN '10-Octubre'
                WHEN 11 THEN '11-Noviembre'  WHEN 12 THEN '12-Diciembre'
            END                                         AS mes,
            COUNT(*)                                    AS anios_observados,
            ROUND(AVG(new_customers), 1)                AS promedio_clientes,
            MIN(new_customers)                          AS minimo,
            MAX(new_customers)                          AS maximo,
            ROUND(STDDEV(new_customers), 1)             AS desviacion_estandar,
            -- Coeficiente de variación: stddev/mean. Bajo = estacional, Alto = aleatorio
            ROUND(STDDEV(new_customers) / NULLIF(AVG(new_customers), 0) * 100, 1)
                                                        AS coef_variacion_pct,
            ROUND(AVG(mom_growth_pct), 1)               AS mom_growth_promedio_pct
        FROM por_mes_anio
        GROUP BY month_number
    ),
    tendencia_anual AS (
        SELECT
            year,
            SUM(new_customers)           AS total_anual,
            ROUND(AVG(new_customers), 1) AS promedio_mensual
        FROM por_mes_anio
        GROUP BY year
    )
    SELECT
        e.*,
        -- Comparar cada mes contra el promedio global para detectar picos
        ROUND(e.promedio_clientes - AVG(e.promedio_clientes) OVER (), 1)
                                                        AS desvio_vs_media_global
    FROM estacionalidad e
    ORDER BY e.month_number
""")

display(df_estacionalidad)

### Hallazgos

**Sin estacionalidad detectable:** El coeficiente de variación por mes (~12%) es bajo
y uniforme — ningún mes muestra consistentemente mayor o menor adquisición entre los
6 años observados. La columna `desvio_vs_media_global` confirma que todos los meses
oscilan alrededor de la media global (~68 clientes) sin picos sistemáticos.

En una institución real en LATAM esperaríamos:
- **Enero:** pico por resoluciones de año nuevo y desembolso de bonos anuales
- **Marzo-Abril:** segundo pico pre-Semana Santa
- **Noviembre-Diciembre:** pico por campañas de fin de año y aguinaldos
- **Julio-Agosto:** valle por vacaciones de invierno en el Cono Sur

La ausencia de estos patrones confirma que `registration_date` fue asignada
aleatoriamente uniforme entre 2020 y 2026 por el generador sintético.

**Sin tendencia interanual:** Los totales anuales son muy similares (~820 clientes/año),
sin la curva de crecimiento acelerado que caracterizaría a un fintech en expansión.

**Artefacto de corte de mayo 2026:** Solo 17 clientes vs el promedio de 68. No es
una caída real — es el momento de generación del dataset. Este mismo artefacto motivó
el fix de `date_trunc('month', current_date)` en los marts de revenue para excluir
el mes en curso de las agregaciones temporales (documentado en `decisions.md`).

---
# Resumen Ejecutivo

| Dashboard | Pregunta | Hallazgo | Limitación dataset |
|---|---|---|---|
| **Revenue & Transactions** | ¿Qué canal es más confiable? | Fallo ~25% uniforme en todos los canales. Sin diferenciación por canal. | Estados distribuidos uniformemente (real < 3% fallo) |
| **Risk & Credit** | ¿Dónde se concentra el riesgo? | Delinquency ~63% uniforme entre segmentos. 46% clientes en Poor. FICO no predice. | Sin correlación segmento-riesgo ni score-delinquencia |
| **Customer & Engagement** | ¿Hay estacionalidad? | CV ~12% uniforme, sin picos estacionales ni tendencia interanual. | Fechas asignadas aleatoriamente por el generador |

Los tres hallazgos convergen en la misma conclusión de `decisions.md`:
el generador distribuyó las variables financieras **sin correlación entre dimensiones**.
El pipeline captura y expone fielmente esa realidad sin enmascarar los artefactos —
lo que demuestra que los 434 tests de dbt detectarían anomalías en datos reales de producción.